# Layer 06 — RAG Property Search (TinyLlama)

Demonstrates a 3-stage **Retrieval-Augmented Generation** pipeline for natural-language HDB property search:

| Stage | What happens |
|-------|--------------|
| **1. NL → Search Params** | TinyLlama extracts `{ weights, filters }` JSON from free-text query |
| **2. Search Execution** | `Neo4jPropertySearch.search()` runs Cypher → top-k property results |
| **3. Results → NL Answer** | TinyLlama summarises the results into a plain-English answer |

**Prerequisites:**
- Run `01_build_knowledge_base.ipynb` to push the property KB to Neo4j
- Neo4j credentials in `.env` at repo root (`NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`)

**Required packages** (install if not already present in your environment):
```
transformers>=4.36.0
torch>=2.0.0
accelerate>=0.26.0
langchain>=0.1.0
langchain-huggingface>=0.0.3
neo4j
python-dotenv
```

In [36]:
# --- Path setup (standard PropertyLens pattern) ---
import sys
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / 'hf_data').exists() else cwd.parent
LAYER_DIR = REPO_ROOT / '06_search_layer'

# Make layer module importable
if str(LAYER_DIR) not in sys.path:
    sys.path.insert(0, str(LAYER_DIR))

load_dotenv(REPO_ROOT / '.env')

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'LAYER_DIR : {LAYER_DIR}')

REPO_ROOT : /Users/lorenzolou/VScode/PropertyLens
LAYER_DIR : /Users/lorenzolou/VScode/PropertyLens/06_search_layer


## 1. Load TinyLlama

This cell loads the TinyLlama-1.1B-Chat model (~2.2 GB download on first run). 
The `local_llm` object is passed to `PropertyRAGSearch` to avoid double-loading.

In [37]:
# 5. Initialize Local LLM (Using TinyLlama - simpler and no gated access required)
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
from langchain_huggingface import HuggingFacePipeline

# TinyLlama is a great lightweight alternative for testing
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15
)

local_llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("TinyLlama initialized successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TinyLlama initialized successfully!


## 2. Initialise PropertyRAGSearch

Pass the already-loaded `local_llm` to avoid loading TinyLlama a second time.
Neo4j credentials are read from `.env` automatically.

In [50]:
import importlib
import yc_property_search
import yc_rag_search

importlib.reload(yc_property_search)
importlib.reload(yc_rag_search)

from yc_rag_search import PropertyRAGSearch

rag = PropertyRAGSearch(llm=local_llm)
print(rag)

PropertyRAGSearch(model='TinyLlama/TinyLlama-1.1B-Chat-v1.0', top_k=5)


## 3. Stage 1 Isolated Test — Parameter Extraction

Before running the full pipeline, verify that TinyLlama correctly extracts `weights` and `filters` from free-text queries.

In [41]:
test_queries = [
    "What are the top primary schools near 1 Lorong Lew Lian Serangoon? Which are the most competitive and what are their quality tiers?",
]

for q in test_queries:
    params, fallback, raw = rag._stage1_extract_params(q)
    print(f"Query   : {q}")
    print(f"Params  : {json.dumps(params, indent=2)}")
    print(f"Fallback: {fallback}")
    if fallback:
        print(f"Raw out : {raw[:120]!r}")
    print()

Query   : What are the top primary schools near 1 Lorong Lew Lian Serangoon? Which are the most competitive and what are their quality tiers?
Params  : {
  "weights": {
    "score_famous_school": 8.0
  },
  "filters": {
    "town": "SERANGOON",
    "address_key": "1 LORONG LEW LIAN"
  },
  "special_query_type": null,
  "cypher_hint": null
}
Fallback: True
Raw out : 'Sure! Here\'s an example with more detailed information for each key word:\n\nQuery: "Show me flats around 1 Lorong Lew Ser'



## 4. End-to-End Scenarios

Each cell runs the full 3-stage pipeline and displays:
- The extracted search params (Stage 1)
- The top matching properties (Stage 2)
- The plain-English answer (Stage 3)

In [52]:
# --- Scenario 1: Education-focused family ---
result = rag.ask("What are the top primary schools near 1 Lorong Lew Lian Serangoon? Which are the most competitive and what are their quality tiers?", top_k=100)

print("=" * 60)
print("SCENARIO 1 — Education-focused family")
print("=" * 60)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nFallback used: {result['fallback_used']}")
print()

display_cols = [c for c in [
    'address_key', 'town', 'flat_type', 'floor_area_sqm',
    'resale_price', 'dist_to_nearest_famous_school_km',
    'nearest_famous_school_name', 'composite_score'
] if c in result['results'].columns]
display(result['results'][display_cols])

Unable to retrieve routing information
/Users/lorenzolou/VScode/PropertyLens/06_search_layer/yc_rag_search.py:832: UserWarning: Neo4j search failed in Stage 2: Unable to retrieve routing information
  if not records:
Unable to retrieve routing information
/Users/lorenzolou/VScode/PropertyLens/06_search_layer/yc_rag_search.py:832: UserWarning: Neo4j search failed in Stage 2: Unable to retrieve routing information
  if not records:


SCENARIO 1 — Education-focused family

Extracted params:
{
  "weights": {
    "score_famous_school": 8.0
  },
  "filters": {
    "town": "SERANGOON",
    "address_key": "1 LORONG LEW LIAN"
  },
  "special_query_type": null,
  "cypher_hint": null
}

Answer:
Top matches: 1 LOR LEW LIAN (SERANGOON, 3 ROOM, SGD 438,000); 2 LOR LEW LIAN (SERANGOON, 3 ROOM, SGD 445,000); 3 LOR LEW LIAN (SERANGOON, 3 ROOM, SGD 488,888).

Fallback used: True



,address_key,town,flat_type,floor_area_sqm,resale_price,dist_to_nearest_famous_school_km,nearest_famous_school_name,composite_score
0,1 LOR LEW LIAN,SERANGOON,3 ROOM,64.00,"438,000.00",2.45,ROSYTH SCHOOL,7.85
1,2 LOR LEW LIAN,SERANGOON,3 ROOM,64.00,"445,000.00",2.48,ROSYTH SCHOOL,7.82
2,3 LOR LEW LIAN,SERANGOON,3 ROOM,73.00,"488,888.00",2.49,ROSYTH SCHOOL,7.80
3,4 LOR LEW LIAN,SERANGOON,3 ROOM,73.00,"460,000.00",2.52,ROSYTH SCHOOL,7.77
4,5 LOR LEW LIAN,SERANGOON,3 ROOM,64.00,"412,000.00",2.55,ROSYTH SCHOOL,7.74
...,...,...,...,...,...,...,...,...
95,111 SERANGOON NTH AVE 1,SERANGOON,4 ROOM,91.00,"585,000.00",0.63,ROSYTH SCHOOL,9.76
96,143 SERANGOON NTH AVE 1,SERANGOON,5 ROOM,122.00,"750,000.00",0.64,ROSYTH SCHOOL,9.75
97,113 SERANGOON NTH AVE 1,SERANGOON,3 ROOM,67.00,"380,000.00",0.66,ROSYTH SCHOOL,9.73
98,117 SERANGOON NTH AVE 1,SERANGOON,5 ROOM,121.00,"685,000.00",0.70,ROSYTH SCHOOL,9.70


In [ ]:
# --- Scenario 2: Commuter with budget constraint ---
result = rag.ask("3-room flat near MRT in Toa Payoh under $500k", top_k=5)

print("=" * 60)
print("SCENARIO 2 — Commuter with budget constraint")
print("=" * 60)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"\nAnswer:\n{result['answer']}")
print()

display_cols = [c for c in [
    'address_key', 'town', 'flat_type', 'resale_price',
    'dist_to_mrt_m', 'score_mrt', 'composite_score'
] if c in result['results'].columns]
display(result['results'][display_cols])

In [ ]:
# --- Scenario 3: Graph traversal — near specific famous school ---
result = rag.ask("Show me flats within 1km of Nanyang Primary School", top_k=8)

print("=" * 60)
print("SCENARIO 3 — Graph traversal (near specific school)")
print("=" * 60)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"Special query type: {result['params'].get('special_query_type')}")
print(f"\nAnswer:\n{result['answer']}")
print()

if not result['results'].empty:
    display(result['results'])
else:
    print("(No results — check Neo4j NEAR_FAMOUS_SCHOOL relationships)")

In [ ]:
# --- Scenario 4: Lifestyle criteria — quiet, large, good value ---
result = rag.ask(
    "Large 5-room flat with long lease, affordable price, away from highway noise",
    top_k=5
)

print("=" * 60)
print("SCENARIO 4 — Lifestyle criteria (quiet, large, value)")
print("=" * 60)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"\nAnswer:\n{result['answer']}")
print()

display_cols = [c for c in [
    'address_key', 'town', 'flat_type', 'floor_area_sqm',
    'lease_remaining_years', 'resale_price',
    'score_quietness', 'score_size', 'score_lease', 'composite_score'
] if c in result['results'].columns]
display(result['results'][display_cols])

## 5. Error Handling

Verify that the pipeline degrades gracefully for ambiguous queries and impossible filters.

In [ ]:
# --- Edge case 1: Ambiguous / very short query → should trigger fallback ---
result = rag.ask("???", top_k=5)
print(f"Query      : ???")
print(f"Fallback   : {result['fallback_used']}")
print(f"Error msg  : {result['error']}")
print(f"Answer     : {result['answer']}")
print()

# --- Edge case 2: Impossible budget (no property this cheap) → empty results ---
result = rag.ask("3-room flat in Bishan under $100k", top_k=5)
print(f"Query      : 3-room flat in Bishan under $100k")
print(f"Empty      : {result['results'].empty}")
print(f"Answer     : {result['answer']}")

## 6. Interactive Search

Edit the `user_query` variable below and re-run the cell.

In [ ]:
# ✏️  Edit this query and re-run the cell
user_query = "Find a 4-room flat near Tampines with famous school and MRT access"

result = rag.ask(user_query, top_k=5)

print(f"Query  : {user_query}")
print(f"\nExtracted params:")
print(json.dumps(result['params'], indent=2))
print(f"\nAnswer:\n{result['answer']}")
print(f"\nFallback used: {result['fallback_used']}")
print()
display(result['results'])

In [ ]:
# --- Cleanup ---
rag.close()
print("Neo4j connection closed.")